In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 3 — O DILEMA DE HIPÓCRATES: FALSOS POSITIVOS VS FALSOS NEGATIVOS

> "Primeiro, não causar dano. Uma regra simples para um médico, um dilema complexo para um algoritmo. Pois o dano pode vir tanto da ação desnecessária quanto da inação fatal."
> — Reflexão sobre o Juramento de Hipócrates no século XXI

Me lembrei de uma conversa com o chefe da oncologia, pouco antes do lançamento do **OncoClassify 1.0**. "O mais importante", disse ele, ajustando os óculos, "é não deixar passar nada. Um alarme falso é um inconveniente; uma ameaça não detectada é uma tragédia." Na época concordei sem entender o peso de suas palavras. Para mim, um erro era um erro. Um número a ser minimizado.

Hoje, as palavras dele ecoam com clareza dolorosa. O "inconveniente" de um **Falso Positivo** leva a mais exames, talvez uma biópsia, semanas de ansiedade — mas termina em alívio. A "tragédia" de um **Falso Negativo** leva a tratamento tardio, a uma janela perdida, a consequências irreversíveis.

Meu trabalho agora não era só construir um classificador, mas incorporar a sabedoria de Hipócrates em linhas de código: ensinar ao modelo que o custo dos erros não é simétrico. O dilema não era mais só técnico, era ético — o *trade-off* fundamental da medicina diagnóstica, entre **Precisão** e **Recall**.

## O Trade-Off Entre Precisão e Recall

É raro, quase impossível, maximizar Precisão e Recall ao mesmo tempo. Melhorar uma geralmente degrada a outra — essa relação inversa é o ***trade-off* Precisão-Recall**.

**Para aumentar a Precisão**, o modelo fica mais cauteloso: só rotula "Maligno" com altíssima certeza. Reduz Falsos Positivos, mas deixa passar casos ambíguos, aumentando os Falsos Negativos.

**Para aumentar o Recall**, o modelo fica mais agressivo: rotula "Maligno" com evidência fraca. Reduz Falsos Negativos, mas gera mais Falsos Positivos.

| Cenário | Foco | Consequência positiva | Consequência negativa |
|---|---|---|---|
| Maximizar Precisão | Evitar Falsos Positivos | Menos biópsias e ansiedade | Risco de não diagnosticar doentes |
| Maximizar Recall | Evitar Falsos Negativos | Maior chance de achar todos os doentes | Mais alarmes falsos |

No diagnóstico de câncer, o consenso é claro: **maximizar o Recall**. É preferível investigar dez alarmes falsos a deixar um único câncer passar. Muitos algoritmos não decidem a classe diretamente — produzem uma **probabilidade**, comparada a um **limiar (threshold)**, por padrão 0,5. Ajustar esse limiar é o que nos permite navegar o *trade-off* — tema que retomaremos, com código, no Capítulo 16.

> **📘 PARA SABER — Sensibilidade e Especificidade: a linguagem da medicina**
>
> **Sensibilidade** = VP / (VP + FN) — idêntica ao **Recall**. Mede a capacidade de identificar corretamente os doentes.
>
> **Especificidade** = VN / (VN + FP) — mede a capacidade de identificar corretamente os saudáveis.
>
> Para rastreamento de câncer, o objetivo é **alta sensibilidade (alto Recall)**.

## Quantificando o Fracasso do Baseline

Vamos calcular formalmente todas as métricas do nosso modelo de base. Isso solidifica os conceitos e prepara o terreno para avaliar modelos reais. Reaproveitamos o alvo canônico e o mesmo baseline do capítulo anterior — sem recriar nada, fiéis ao princípio DRY.

#### Código 3.1: Métricas do Modelo de Base


In [ ]:
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, confusion_matrix)
from oncoclassify_utils import y   # 0 = Maligno (classe positiva), 1 = Benigno

y_pred_base = np.ones_like(y)      # baseline: sempre Benigno

# pos_label=0 -> as metricas sao calculadas PARA a classe Maligna.
# zero_division=0 -> quando o modelo nunca preve Maligno, retorna 0 em vez de erro.
acc  = accuracy_score(y, y_pred_base)
prec = precision_score(y, y_pred_base, pos_label=0, zero_division=0)
rec  = recall_score(y, y_pred_base, pos_label=0, zero_division=0)

# Especificidade nao tem funcao pronta: derivamos da matriz de confusao.
cm = confusion_matrix(y, y_pred_base, labels=[0, 1])
vn, fp = cm[1, 1], cm[1, 0]
espec = vn / (vn + fp)

print(f"Acuracia:        {acc:.4f}")
print(f"Precisao:        {prec:.4f}")
print(f"Recall (Sens.):  {rec:.4f}")
print(f"Especificidade:  {espec:.4f}")

Acuracia:        0.6333
Precisao:        0.0000
Recall (Sens.):  0.0000
Especificidade:  1.0000


Os números descrevem o comportamento do baseline com precisão cirúrgica:

**Acurácia (0,6333)** — enganosa, como já sabemos.

**Precisão (0,0000)** — nunca previu "Maligno", nunca acertou essa previsão.

**Recall (0,0000)** — não encontrou **nenhum** dos 220 cânceres. É o número da falha catastrófica.

**Especificidade (1,0000)** — perfeito em identificar benignos, pois era tudo o que previa.

Um modelo pode ser perfeito numa métrica (Especificidade) e um fracasso total em outra (Recall). É o equilíbrio entre elas que define um classificador útil.

### F1-Score: Uma Média Harmônica

E se quisermos uma única métrica que leve em conta Precisão e Recall? A **média harmônica** de ambas — o **F1-Score**:

**F1 = 2 · (Precisão · Recall) / (Precisão + Recall)**

A média harmônica pende para o menor dos dois valores, então **pune extremos**: só é alta se Precisão *e* Recall forem altos. Para o baseline, com ambos em zero, o F1 também é **0**.

> **💡 DICA — Quando usar o F1-Score?**
>
> O F1 é útil quando as classes são desbalanceadas e os custos de FP e FN são parecidos. No nosso caso, como o custo de um Falso Negativo é muito maior, o **Recall** é a prioridade — mas o F1 continua sendo um excelente termômetro geral do equilíbrio do modelo.

> **📌 CHECKLIST DO CAPÍTULO 3**
>
> ✅ Entende o *trade-off* Precisão-Recall: melhorar uma geralmente piora a outra.
>
> ✅ Compreende como o **limiar de decisão** controla esse *trade-off*.
>
> ✅ Sabe que **Sensibilidade = Recall** e o que é a **Especificidade**.
>
> ✅ Executou o Código 3.1 e viu **0** para Precisão e Recall e **1** para Especificidade no baseline.
>
> ✅ Sabe o que é o **F1-Score** e por que ele pune extremos.
>
> **⚠️ CRÍTICO:** Escolher qual métrica otimizar (Precisão, Recall, F1) não é decisão técnica, é decisão clínica — guiada pelo custo relativo de cada tipo de erro.

Os números na tela eram frios, mas a clareza que traziam era reconfortante. Eu finalmente tinha um *framework* para articular o dilema ético que o chefe da oncologia me apresentara. Não se tratava de eliminar o erro, mas de escolher qual erro estávamos dispostos a tolerar. E no diagnóstico de câncer, toleramos o inconveniente do alarme falso para evitar a catástrofe da ameaça silenciosa.

Com a base filosófica estabelecida, a primeira parte da jornada estava completa. Agora era hora de arregaçar as mangas e preparar o campo de batalha: **os dados**. E a primeira lei dessa nova fase seria a mais sagrada de todas — a separação inviolável entre treino e teste.

**PARTE II — PREPARANDO O CAMPO DE BATALHA**
